# register-back-fn-after-wrap — ex1: wire one entry into BACK_FUNCS and dispatch it

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `register-back-fn-after-wrap`. Running the final beacon cell reports progress against the `Backprop: register back fn` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: register back fn` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`register-back-fn-after-wrap`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "register-back-fn-after-wrap"
DD_SUBTOPIC = "Backprop: register back fn"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Register-back-fn-after-wrap — quick refresher

A tiny autograd needs to look up *which* backward fn corresponds to *which* forward op at *which* argument position. The convention is a dict keyed by `(forward_fn, argnum)`:

```python
class BackwardFuncLookup:
    def __init__(self): self._table = {}
    def add_back_func(self, fwd, argnum, back_fn):
        self._table[(fwd, argnum)] = back_fn
    def get_back_func(self, fwd, argnum):
        return self._table[(fwd, argnum)]

BACK_FUNCS = BackwardFuncLookup()
log = wrap_forward_fn(torch.log)
BACK_FUNCS.add_back_func(torch.log, 0, log_back)   # the wrap+register pair
```

Two steps, always in this order: (1) wrap the forward fn so it builds a Recipe; (2) register the backward fn so the reverse pass can find it. Binary ops register TWICE — once for argnum=0, once for argnum=1.

### Exercise 1 — wire one entry into BACK_FUNCS and dispatch it

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the (forward_fn, argnum) → back_fn registration pattern by wiring a single entry into a BackwardFuncLookup table and dispatching through it.
> Keywords: back-funcs, dict-by-tuple, register, dispatch
> ```

**KCs targeted:** `register-back-fn-after-wrap`, `backward-func-lookup`

Implement TWO things:

**1. `BackwardFuncLookup` class** with two methods:
   - `add_back_func(fwd_fn, argnum, back_fn)` — register a back fn.
   - `get_back_func(fwd_fn, argnum)` — look one up.
   Use a dict keyed by `(fwd_fn, argnum)` tuples internally.

**2. `register_log(BACK_FUNCS)`** — given a fresh lookup table, register the backward fn for `torch.log` at argnum=0. The backward fn is provided to you as `log_back(grad_out, out, x) -> grad_out / x`.

The test cell then **dispatches** through the table: it looks up `(torch.log, 0)` and calls the returned back fn with concrete args, verifying both the registration and the lookup work.

**Why this is the bedrock of the autograd dispatcher.** Once the table is populated, the reverse pass becomes a generic loop: pop a Tensor off the topo-sorted queue, look up its `(recipe.func, argnum)`, call the back fn. No `if fn == log: ...` chains. This drill is the single-entry baby step.

In [ ]:
class BackwardFuncLookup:
    """Dict-keyed-by-(fwd_fn, argnum) lookup table for backward fns."""
    def __init__(self):
        raise NotImplementedError()

    def add_back_func(self, fwd_fn, argnum, back_fn):
        raise NotImplementedError()

    def get_back_func(self, fwd_fn, argnum):
        raise NotImplementedError()


def log_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    return grad_out / x


def register_log(BACK_FUNCS: 'BackwardFuncLookup') -> None:
    """Register log_back for torch.log at argnum=0."""
    raise NotImplementedError()


def _test_ex1():
    # --- lookup table sanity ---
    tbl = BackwardFuncLookup()

    def fake_back(grad_out, out, x):
        return grad_out * 2

    tbl.add_back_func(t.sin, 0, fake_back)
    got = tbl.get_back_func(t.sin, 0)
    assert got is fake_back, f'add/get round-trip failed: got {got}'

    # Different argnums for the same fwd_fn must be independent slots.
    def fake_back_arg1(grad_out, out, x, y):
        return grad_out * 3

    tbl.add_back_func(t.multiply, 0, fake_back)
    tbl.add_back_func(t.multiply, 1, fake_back_arg1)
    assert tbl.get_back_func(t.multiply, 0) is fake_back
    assert tbl.get_back_func(t.multiply, 1) is fake_back_arg1

    # --- register_log + dispatch ---
    BACK_FUNCS = BackwardFuncLookup()
    register_log(BACK_FUNCS)

    # Pull the registered fn back out — must equal log_back.
    fn = BACK_FUNCS.get_back_func(t.log, 0)
    assert fn is log_back, f'expected log_back, got {fn}'

    # End-to-end dispatch: simulate the reverse pass for a single log node.
    x = t.tensor([1.0, 2.0, 4.0])
    out = t.log(x)
    grad_out = t.ones(3)
    # This is what backprop's inner loop does:
    back_fn = BACK_FUNCS.get_back_func(t.log, 0)
    grad_x = back_fn(grad_out, out, x)
    expected = t.tensor([1.0, 0.5, 0.25])
    assert t.allclose(grad_x, expected), f'dispatch fail: {grad_x}'

    # Looking up an unregistered (fn, argnum) must raise — silent return
    # would mask bugs in real backprop.
    try:
        BACK_FUNCS.get_back_func(t.exp, 0)
        raised = False
    except (KeyError, LookupError):
        raised = True
    assert raised, 'get_back_func on missing (fn, argnum) should raise'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
class BackwardFuncLookup:
    def __init__(self):
        # Dict keyed by (fwd_fn, argnum). Tuples hash fine since
        # functions are hashable by identity.
        self._table = {}

    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn

    def get_back_func(self, fwd_fn, argnum):
        # KeyError naturally propagates on miss — that's the right
        # failure mode (a Recipe references an unwrapped op).
        return self._table[(fwd_fn, argnum)]


def log_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    return grad_out / x


def register_log(BACK_FUNCS: 'BackwardFuncLookup') -> None:
    BACK_FUNCS.add_back_func(t.log, 0, log_back)
```

**Why a tuple key and not nested dicts.** `{(fwd, argnum): fn}` is a single hash lookup. `{fwd: {argnum: fn}}` is two lookups and an extra missing-key branch. ARENA uses the flat tuple form everywhere — it's the same data model as PyTorch's C++ Node::next_edges, and pickle/repr are nicer.

**Why KeyError on miss is correct.** If backprop encounters a Recipe whose forward fn was never wrapped/registered, that's a build-time bug (someone forgot a `wrap_forward_fn` call). Raising loud beats returning `None` and exploding ten frames deep with `'NoneType' is not callable`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()